# Universal QR Code Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Favioleiva/UsefulJupyterNotebooks/blob/main/Universal_QR_Code_Generator.ipynb)

**GitHub:** [https://github.com/Favioleiva/UsefulJupyterNotebooks/blob/main/Universal_QR_Code_Generator.ipynb](https://github.com/Favioleiva/UsefulJupyterNotebooks/blob/main/Universal_QR_Code_Generator.ipynb)  
**Open directly in Google Colab:** [https://colab.research.google.com/github/Favioleiva/UsefulJupyterNotebooks/blob/main/Universal_QR_Code_Generator.ipynb](https://colab.research.google.com/github/Favioleiva/UsefulJupyterNotebooks/blob/main/Universal_QR_Code_Generator.ipynb)

A small, reusable Jupyter Notebook that converts **any web link into a QR code**.

**Author:** Favio Sergio Leiva Cárdenas  
**Co-author / AI assistant:** OpenAI ChatGPT (GPT-5.6 Sol)

### How to use
1. Run the installation cell once.
2. Run the QR generator cell.
3. Paste any web link when prompted.
4. The notebook immediately displays the QR code and saves both **PNG** and **SVG** versions.

The notebook is intentionally generic so it can be reused for websites, papers, datasets, appendices, repositories, forms, or any other URL.

In [ ]:
# Install the only external dependency.
# Safe to re-run: pip will skip installation if the package is already available.
%pip install -q "qrcode[pil]"

## Generate a QR code

Run the cell below and paste the link you want to encode.

Example:

`https://example.com/`

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import re

import qrcode
import qrcode.image.svg
from IPython.display import display, Image, FileLink, Markdown


def normalize_url(raw_url: str) -> str:
    """Clean and validate a user-supplied web link."""
    url = raw_url.strip()

    if not url:
        raise ValueError("No URL was provided.")

    # Make common pasted links friendlier: add https:// when omitted.
    if "://" not in url:
        url = "https://" + url

    parsed = urlparse(url)

    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError(
            "Please provide a valid web URL, for example: https://example.com/"
        )

    return url


def safe_filename_from_url(url: str) -> str:
    """Create a short filesystem-safe name from the URL host/path."""
    parsed = urlparse(url)
    base = (parsed.netloc + parsed.path).strip("/") or "link"
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_.")
    return (base[:80] or "link")


def generate_qr(url: str, output_dir: str = "."):
    """
    Generate publication-friendly QR codes in PNG and SVG formats.

    Error correction H allows the QR code to remain readable even if
    a portion is damaged or slightly obscured.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    stem = safe_filename_from_url(url)
    png_path = output_dir / f"QR_{stem}.png"
    svg_path = output_dir / f"QR_{stem}.svg"

    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=20,
        border=4,
    )
    qr.add_data(url)
    qr.make(fit=True)

    # PNG: convenient for documents, slides, and websites.
    png_img = qr.make_image(fill_color="black", back_color="white")
    png_img.save(png_path)

    # SVG: vector version for LaTeX, print, or resizing without quality loss.
    svg_img = qr.make_image(image_factory=qrcode.image.svg.SvgPathImage)
    svg_img.save(svg_path)

    return png_path, svg_path


# ---- USER ACTION: paste a link when prompted ----
raw_link = input("Paste the web link to encode as a QR code: ")
url = normalize_url(raw_link)

png_file, svg_file = generate_qr(url)

display(Markdown(f"### QR code generated\n**Encoded URL:** `{url}`"))
display(Image(filename=str(png_file), width=360))

print("\nSaved files:")
display(FileLink(str(png_file)))
display(FileLink(str(svg_file)))

## Notes for reuse and publication

- **PNG** is convenient for Word, PowerPoint, HTML, and most document workflows.
- **SVG** is preferable when you want lossless scaling for print or LaTeX.
- The QR code uses **high error correction (Level H)**.
- Always verify the encoded destination before publishing a QR code permanently.
- For archival documents, prefer a stable URL that you control.

---
### Suggested citation / credit

**Leiva Cárdenas, Favio Sergio & OpenAI ChatGPT (GPT-5.6 Sol).**  
*Universal QR Code Generator*. Jupyter Notebook.